<a href="https://colab.research.google.com/github/Shalini-priya6299/nl2sql-spider/blob/part1-data-eval-zeroshot/notebooks/run_e1_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# E1: Zero-shot Text-to-SQL on Spider

Runtime → Change runtime type → **T4 GPU**. Run the cells in order.
Results are saved in `results/` and copied to Google Drive so they survive a disconnect.

In [1]:
import os
import subprocess
from google.colab import userdata

BRANCH = "part1-data-eval-zeroshot"
REPO = "Shalini-priya6299/nl2sql-spider"

token = userdata.get("GITHUB_TOKEN")

# Create a temporary Git authentication helper.
askpass = "/tmp/github-askpass.sh"
with open(askpass, "w") as f:
    f.write("""#!/bin/sh
case "$1" in
  *Username*) echo "x-access-token" ;;
  *Password*) echo "$GITHUB_TOKEN" ;;
esac
""")
os.chmod(askpass, 0o700)

env = os.environ.copy()
env["GITHUB_TOKEN"] = token
env["GIT_ASKPASS"] = askpass
env["GIT_TERMINAL_PROMPT"] = "0"

subprocess.run(
    ["git", "clone", "-q", "-b", BRANCH,
     f"https://github.com/{REPO}.git", "nl2sql-spider"],
    env=env,
    check=True
)

os.remove(askpass)
del token, env

print("Repository cloned successfully!")

Repository cloned successfully!


In [2]:
# Optional: keep results on Google Drive (resume works if Colab disconnects)
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/nl2sql-results && rm -rf results && ln -s /content/drive/MyDrive/nl2sql-results results

MessageError: Error: credential propagation was unsuccessful

In [ ]:
!python data_utils.py download --only-dev
!python data_utils.py stats
!python -m pytest -q tests/

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python run_e1_zeroshot.py --subset balanced --batch-size 8

In [ ]:
# Full dev set (1,034 questions) for the headline number
!python run_e1_zeroshot.py --subset full --batch-size 8

In [ ]:
RUN = 'results/e1_zeroshot_Qwen2.5-Coder-1.5B-Instruct_balanced'
!python error_analysis.py sample $RUN/eval_details.jsonl -n 25

Download `error_analysis.csv`, fill in the **category** column (allowed values are printed above), put it back in the same folder, then run:

In [ ]:
!python error_analysis.py summarize $RUN/error_analysis.csv
!python make_results.py --labels e1_zeroshot_Qwen2.5-Coder-1.5B-Instruct_balanced='E1 zero-shot'